# Notebook 02_04 — Feature Selection: SHAP

Runs **SHAP** across K = {4, 8, 16} × 6 classifiers × 5 seeds.
Uses the pre-computed ranking from `02_00_fs_rankings.ipynb` (`models/fs_rankings.pkl`).

**Results saved incrementally to** `results/fs_shap_raw.csv` — if this notebook crashes mid-run, just re-run it: completed (K, seed, classifier) combinations are skipped automatically.

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)
Imports OK


In [2]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']
y_train = d['y_train']
FEATURE_NAMES = d['feature_names']

with open(f'{MODELS_DIR}fs_rankings.pkl', 'rb') as f:
    RANKINGS = pickle.load(f)

METHOD_NAME = 'SHAP'
ranked_indices = RANKINGS[METHOD_NAME]

print(f'Method: {METHOD_NAME}')
print('Top-16 features:', [FEATURE_NAMES[i] for i in ranked_indices[:16]])

Method: SHAP
Top-16 features: ['tcp.dstport', 'arp.opcode', 'radiotap.channel.freq', 'frame.len', 'tcp.srcport', 'radiotap.mactime', 'wlan_radio.duration', 'radiotap.length', 'frame.encap_type', 'wlan_radio.signal_dbm', 'tcp.time_relative', 'wlan.seq', 'ip.proto', 'wlan.duration', 'wlan.fc.subtype', 'tcp.flags.syn']


## Transform function (slice to top-K features by this method's ranking)

In [3]:
def transform_fn(K):
    selected = ranked_indices[:K]
    return X_train[:, selected], X_val[:, selected], X_test[:, selected], K

## Run experiment grid (resumable)

In [4]:
RESULTS_CSV = f'{RESULTS_DIR}fs_shap_raw.csv'

run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

[SHAP] K=4 seed=42 DT     F1=0.7781  train=0.1s  inf=0.0001ms/sample  (1/90)
[SHAP] K=4 seed=42 RF     F1=0.7970  train=1.3s  inf=0.0029ms/sample  (2/90)
[SHAP] K=4 seed=42 XGB    F1=0.7889  train=1.9s  inf=0.0012ms/sample  (3/90)
[SHAP] K=4 seed=42 LGBM   F1=0.0732  train=1.1s  inf=0.0011ms/sample  (4/90)
[SHAP] K=4 seed=42 KNN    F1=0.5601  train=0.1s  inf=0.0813ms/sample  (5/90)
[SHAP] K=4 seed=42 MLP    F1=0.6995  train=37.2s  inf=0.0022ms/sample  (6/90)
[SHAP] K=4 seed=52 DT     F1=0.7795  train=0.0s  inf=0.0001ms/sample  (7/90)
[SHAP] K=4 seed=52 RF     F1=0.7917  train=1.3s  inf=0.0027ms/sample  (8/90)
[SHAP] K=4 seed=52 XGB    F1=0.7785  train=1.9s  inf=0.0012ms/sample  (9/90)
[SHAP] K=4 seed=52 LGBM   F1=0.7905  train=1.2s  inf=0.0036ms/sample  (10/90)
[SHAP] K=4 seed=52 KNN    F1=0.5603  train=0.1s  inf=0.0666ms/sample  (11/90)
[SHAP] K=4 seed=52 MLP    F1=0.7032  train=70.4s  inf=0.0021ms/sample  (12/90)
[SHAP] K=4 seed=62 DT     F1=0.7801  train=0.0s  inf=0.0001ms/sample  (

## Quick summary

In [5]:
res_df = pd.read_csv(RESULTS_CSV)
summary = res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std'])
print(summary.to_string())

                     f1           train_time_s            inference_ms              
                   mean       std         mean        std         mean           std
K  classifier                                                                       
4  DT          0.779345  0.000759     0.051013   0.001748     0.000077  1.193206e-06
   KNN         0.554681  0.010369     0.087262   0.004841     0.072605  6.711581e-03
   LGBM        0.381354  0.376440     1.122847   0.084514     0.002333  1.084586e-03
   MLP         0.701666  0.003098    60.737231  19.730497     0.001912  3.181542e-04
   RF          0.794406  0.002313     1.247853   0.035869     0.002740  6.735921e-05
   XGB         0.786219  0.005871     1.828079   0.061661     0.001169  3.536089e-05
8  DT          0.887842  0.001200     0.057523   0.002178     0.000056  1.543604e-06
   KNN         0.737897  0.006331     0.149148   0.003644     0.021697  2.786677e-03
   LGBM        0.412102  0.433536     1.218573   0.123435     0.0